# Talker Comparison — PantoMatrix en Google Colab
Backend batch para generar movimiento con **EMAGE** o **CaMN** sin depender de ZeroGPU.

1. En Colab: `Runtime > Change runtime type > GPU`.
2. Ejecutá las celdas en orden.
3. Subí uno de los audios de Mikves.
4. Elegí modelo y generá.
5. Se descarga un JSON compatible con Talker Comparison.

El Colab gratuito tiene disponibilidad y límites dinámicos; no se usa como servidor permanente, sino como batch interactivo.

In [ ]:
!nvidia-smi
!rm -rf /content/PantoMatrix
!git clone --depth 1 https://github.com/PantoMatrix/PantoMatrix.git /content/PantoMatrix
%cd /content/PantoMatrix
!bash setup.sh

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os

inp = Path('/content/talker_input')
out = Path('/content/talker_output')
shutil.rmtree(inp, ignore_errors=True); shutil.rmtree(out, ignore_errors=True)
inp.mkdir(); out.mkdir()
uploaded = files.upload()
for name, data in uploaded.items():
    (inp/name).write_bytes(data)
print('Audio:', [p.name for p in inp.iterdir()])

In [ ]:
# Elegí: 'EMAGE' o 'CaMN'
MODEL = 'EMAGE'

import subprocess, pathlib
py = '/content/py39/bin/python' if pathlib.Path('/content/py39/bin/python').exists() else 'python'
script = 'test_emage_audio.py' if MODEL.upper() == 'EMAGE' else 'test_camn_audio.py'
cmd = [py, script, '--visualization', '--audio_folder', '/content/talker_input', '--save_folder', '/content/talker_output']
print('Ejecutando:', ' '.join(cmd))
subprocess.run(cmd, cwd='/content/PantoMatrix', check=True)

In [ ]:
# Convierte el NPZ de PantoMatrix al JSON común de Talker Comparison.
import numpy as np, json, math
from pathlib import Path

SMPLX = {'hips':0,'leftUpperLeg':1,'rightUpperLeg':2,'spine':3,'leftLowerLeg':4,'rightLowerLeg':5,'chest':9,'leftFoot':10,'rightFoot':11,'neck':12,'leftShoulder':13,'rightShoulder':14,'head':15,'leftUpperArm':16,'rightUpperArm':17,'leftForeArm':18,'rightForeArm':19,'leftHand':20,'rightHand':21}
def aa_quat(v):
    x,y,z = map(float,v); a=math.sqrt(x*x+y*y+z*z)
    if a < 1e-8: return [0,0,0,1]
    s=math.sin(a/2)/a
    return [x*s,y*s,z*s,math.cos(a/2)]

npzs = list(Path('/content/talker_output').rglob('*.npz'))
if not npzs: npzs = list(Path('/content/PantoMatrix').rglob('*.npz'))
npz_path = max(npzs, key=lambda p:p.stat().st_mtime)
d=np.load(npz_path, allow_pickle=True)
poses=d['poses'] if 'poses' in d else d['motion']
poses=np.asarray(poses)
if poses.ndim>2: poses=poses.reshape(poses.shape[0],-1)
frames=[]
for row in poses:
    joints={}
    for name,idx in SMPLX.items():
        joints[name]={'rotation':[0,0,0,1] if name=='hips' else aa_quat(row[idx*3:idx*3+3])}
    frames.append({'root':{'position':[0,0,0]},'joints':joints})
result={'model':'pantomatrix' if MODEL.upper()=='EMAGE' else 'camn','generator':MODEL+' via Google Colab','fps':30,'frames':frames}
json_path=Path('/content')/f'{MODEL.lower()}-colab-motion.json'
json_path.write_text(json.dumps(result))
print('Fuente:', npz_path, 'frames:', len(frames), 'duración:', round(len(frames)/30,1),'s')
files.download(str(json_path))